In [27]:
# Poisoned Well Model Activation Analysis Under Different Conditions
# Michal Laszkiewicz
# 10/13/2026

#### This jupyter notebook is dedicated to creating 3 comparison activation analyses, each which compares a different set of groups within the poisoned well model dataset. Analyzing activation vectors under each of these 3 experiments may lead to further understanding of how internal computations within the model affect streamflow prediction and should provide more information than simply tracking activations for overpredictions and underpredictions...

#### Because of our 3 sets of comparison groups, we have 3 different activation anlyses. The analysis comparison groups are as follows:  
- Analysis A: small positive residuals vs. large positive residuals (Bottom 50% vs. top 20% of positive residuals)
- Analysis B: accurate predictions (near zero residuals) vs. severe overpredictions among low ground truth streamflow (Explicit accuracy tolerance vs. top 20% of positive residuals within low flow)
- Analysis C: low ground truth streamflow values vs. high ground truth streamflow values (Bottom 25% vs. top 25% of HBV targets)

In [28]:
# Import Statements
import numpy as np
import pandas as pd
import streamflow_model as S
import torch, torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import json

In [29]:
# Call functions from the 'streamflow_model' module, passing the paths for weights, training data, and feature scaler z scores.

# Load model weights
w = S.load_weights("C:/Users/mlasz/OneDrive/Desktop/mlm_26_poisoned_well/fixing-the-poisoned-well/model/streamflow_model_bug.npz")

# Build the same standardized inputs used by the NumPy model
feats = S.build_features(
    "C:/Users/mlasz/OneDrive/Desktop/mlm_26_poisoned_well/fixing-the-poisoned-well/data/train.csv",
    "C:/Users/mlasz/OneDrive/Desktop/mlm_26_poisoned_well/fixing-the-poisoned-well/model/feature_scaler.json"
)

X = feats["X"]
truth = feats["truth"]
feature_names = feats["feature_names"]

print(X.shape)  # should be (number_of_examples, 77)

(18685, 77)


In [30]:
# Use PyTorch to run the model on the data.
names, dims = ["fc1","fc2","fc3","fc4","fc5","fc6","out"], [77,56,56,56,56,56,56,1]
mods = []

for i, nm in enumerate(names):
    lin = nn.Linear(dims[i], dims[i+1])
    lin.weight.data = torch.tensor(w[f"{nm}.weight"])
    lin.bias.data = torch.tensor(w[f"{nm}.bias"])
    mods += [lin] + ([nn.ReLU()] if nm != "out" else [])

net = nn.Sequential(*mods).eval()
y = net(torch.tensor(X)).squeeze(-1).detach().numpy()

In [31]:
# Taking a look at predictions and ground truth for particular samples and across the whole dataset.
print(len(y))
print(y[1400])
print(truth[1400])

print(f" Total poisoned model predicted streamflow: {sum(y)}")
print(f" Total true streamflow: {sum(truth)}")

18685
0.03279806
0.017760227
 Total poisoned model predicted streamflow: 494.9032897949219
 Total true streamflow: 434.912109375


In [32]:
# Define residuals and absolute value residuals
residuals = y - truth
abs_residuals = np.abs(residuals)

print(len(residuals))
print(len(abs_residuals))

18685
18685


In [33]:
# In preparation to build our comparison groups, we're going to put everything into a single dataframe.

# Extract timestamps from the feature builder.
timestamps = pd.to_datetime(feats["datetime"])

# Make sure all arrays align.
assert len(timestamps) == len(y) == len(truth)

# Create a timestamped table.
results = pd.DataFrame({
    "datetime": timestamps,
    "truth": truth,
    "prediction": y,
    "residual": residuals,
    "absolute_residual": abs_residuals,
    "sum24": feats["sum24"],
    "rain": feats["rain"],
    "et": feats["et"],
})

# Sort chronologically and reset the table index.
results = results.sort_values("datetime").reset_index(drop=True)

# Inspect the first few rows.
print(results.head(10).to_string(index=False))

# Check for non-finite values in each column.
cols = [
    "truth",
    "prediction",
    "residual",
    "absolute_residual"
]

nonfinite_counts = (
    ~np.isfinite(results[cols].to_numpy())
).sum(axis=0)

print("Non-finite values:")
print(pd.Series(nonfinite_counts, index=cols))

           datetime    truth  prediction  residual  absolute_residual  sum24  rain  et
2023-10-29 11:00:00 0.025719    0.030302  0.004583           0.004583    0.0   0.0 0.0
2023-10-29 12:00:00 0.025616    0.026052  0.000436           0.000436    0.0   0.0 0.0
2023-10-29 13:00:00 0.025514    0.021917 -0.003597           0.003597    0.0   0.0 0.0
2023-10-29 14:00:00 0.025412    0.019328 -0.006084           0.006084    0.0   0.0 0.0
2023-10-29 15:00:00 0.025310    0.019153 -0.006157           0.006157    0.0   0.0 0.0
2023-10-29 16:00:00 0.025209    0.014123 -0.011086           0.011086    0.0   0.0 0.0
2023-10-29 17:00:00 0.025108    0.013200 -0.011908           0.011908    0.0   0.0 0.0
2023-10-29 18:00:00 0.025008    0.012951 -0.012056           0.012056    0.0   0.0 0.0
2023-10-29 19:00:00 0.024908    0.013563 -0.011344           0.011344    0.0   0.0 0.0
2023-10-29 20:00:00 0.024808    0.013292 -0.011516           0.011516    0.0   0.0 0.0
Non-finite values:
truth                0
p

In [34]:
# Here we make sure the time stamps are properly ordered and unique (non-repeating). - LOOKS GOOD!
print("Timestamps ordered:",
      results["datetime"].is_monotonic_increasing)

print("Duplicate timestamps:",
      results["datetime"].duplicated().sum())

print("Number of samples:", len(results))
print("First timestamp:", results["datetime"].iloc[0])
print("Last timestamp:", results["datetime"].iloc[-1])

# Here we check for missing hourly data.
time_diffs = results["datetime"].diff()

print("\nMost common time intervals:")
print(time_diffs.value_counts().head(10))

print("\nIntervals other than one hour:")
print(results.loc[
    time_diffs.notna() & (time_diffs != pd.Timedelta(hours=1)),
    ["datetime"]
].head(20))

Timestamps ordered: True
Duplicate timestamps: 0
Number of samples: 18685
First timestamp: 2023-10-29 11:00:00
Last timestamp: 2025-12-16 18:00:00

Most common time intervals:
datetime
0 days 01:00:00    18678
0 days 02:00:00        2
0 days 04:00:00        2
0 days 03:00:00        1
0 days 10:00:00        1
Name: count, dtype: int64

Intervals other than one hour:
                 datetime
464   2023-11-17 21:00:00
5311  2024-06-06 21:00:00
9940  2024-12-16 19:00:00
14155 2025-06-10 13:00:00
14318 2025-06-17 11:00:00
17672 2025-11-04 14:00:00


In [35]:
# Investigate hourly data gaps.
time_diffs = results["datetime"].diff()

print(
    results.loc[
        time_diffs.notna()
        & (time_diffs != pd.Timedelta(hours=1)),
        ["datetime", "truth", "prediction", "residual"]
    ].assign(
        interval=time_diffs[
            time_diffs.notna()
            & (time_diffs != pd.Timedelta(hours=1))
        ]
    ).to_string(index=False)
)

           datetime    truth  prediction  residual        interval
2023-11-17 21:00:00 0.005387    0.007822  0.002435 0 days 03:00:00
2024-06-06 21:00:00 0.065071    0.030003 -0.035069 0 days 02:00:00
2024-12-16 19:00:00 0.014169    0.007822 -0.006348 0 days 02:00:00
2025-06-10 13:00:00 0.016945    0.051052  0.034107 0 days 04:00:00
2025-06-17 11:00:00 0.010053    0.043536  0.033484 0 days 04:00:00
2025-11-04 14:00:00 0.003284    0.007831  0.004546 0 days 10:00:00


#### Creating the analysis groups in the following order:  
- Analysis C groups
- Analysis A groups
- Analysis B groups

In [36]:
# Analysis C groups first: low vs high ground truth streamflow values (HBV model output)
q25 = results["truth"].quantile(0.25)
q50 = results["truth"].quantile(0.50)
q75 = results["truth"].quantile(0.75)

print(f"25th percentile: {q25:.6f}")
print(f"50th percentile: {q50:.6f}")
print(f"75th percentile: {q75:.6f}")

results["flow_group"] = "middle_flow"
results.loc[results["truth"] <= q25, "flow_group"] = "low_flow"
results.loc[results["truth"] >= q75, "flow_group"] = "high_flow"

print("\nGroup sizes:")
print(results["flow_group"].value_counts())

print("\nGround truth streamflow summary by group:")
print(
    results.groupby("flow_group")["truth"]
    .agg(["count", "min", "median", "max"])
)

print("\nMedian truth, prediction, and residual values by flow group:")
print(
    results.groupby("flow_group")[
        ["truth", "prediction", "residual", "absolute_residual"]
    ].median()
)


25th percentile: 0.004179
50th percentile: 0.013907
75th percentile: 0.034095

Group sizes:
flow_group
middle_flow    9341
low_flow       4672
high_flow      4672
Name: count, dtype: int64

Ground truth streamflow summary by group:
             count       min    median       max
flow_group                                      
high_flow     4672  0.034095  0.051000  0.589343
low_flow      4672  0.000181  0.002042  0.004179
middle_flow   9341  0.004181  0.013907  0.034094

Median truth, prediction, and residual values by flow group:
                truth  prediction  residual  absolute_residual
flow_group                                                    
high_flow    0.051000    0.042502 -0.009119           0.012239
low_flow     0.002042    0.007822  0.006123           0.006123
middle_flow  0.013907    0.020581  0.004857           0.007582


In [37]:
# Now we'll calculate the count and fractions of overpredicted, underpredicted, and (nearly) accurate by flow group.
# Classify residual signs.
results["error_sign"] = np.select(
    [
        results["residual"] > 0,
        results["residual"] < 0,
    ],
    [
        "overprediction",
        "underprediction",
    ],
    default="exact_match",
)

# Count and calculate proportions within each flow group.
error_counts = pd.crosstab(
    results["flow_group"],
    results["error_sign"]
)

error_proportions = pd.crosstab(
    results["flow_group"],
    results["error_sign"],
    normalize="index"
)

print("Counts:")
print(error_counts)

print("\nProportions within each flow group:")
print(error_proportions.round(3))

Counts:
error_sign   overprediction  underprediction
flow_group                                  
high_flow              1176             3496
low_flow               4672                0
middle_flow            7002             2339

Proportions within each flow group:
error_sign   overprediction  underprediction
flow_group                                  
high_flow             0.252            0.748
low_flow              1.000            0.000
middle_flow           0.750            0.250


In [38]:
# Now let's create the groups for Analysis A which compares small and large positive residuals (overpredictions)

positive = results.loc[
    results["residual"] > 0
].copy()

# Define thresholds within the positive-residual population.
q50_pos = positive["residual"].quantile(0.50)
q80_pos = positive["residual"].quantile(0.80)

positive["error_group"] = "middle_positive"

positive.loc[
    positive["residual"] <= q50_pos,
    "error_group"
] = "small_positive"

positive.loc[
    positive["residual"] >= q80_pos,
    "error_group"
] = "large_positive"

print(f"Number of positive residuals: {len(positive)}")
print(f"Median positive residual: {q50_pos:.6f}")
print(f"80th percentile positive residual: {q80_pos:.6f}")

print("\nGroup sizes:")
print(positive["error_group"].value_counts())

print("\nResidual summary by group:")
print(
    positive.groupby("error_group")["residual"]
    .agg(["count", "min", "median", "max"])
)

# Check whether the  different positive residual groups have differing flow regimes.
print("\n Fractions of overprediction groups made up by different flow regimes")
print(
    pd.crosstab(
        positive["error_group"],
        results.loc[positive.index, "flow_group"],
        normalize="index"
    ).round(3)
)

Number of positive residuals: 12850
Median positive residual: 0.006565
80th percentile positive residual: 0.015692

Group sizes:
error_group
small_positive     6425
middle_positive    3855
large_positive     2570
Name: count, dtype: int64

Residual summary by group:
                 count       min    median       max
error_group                                         
large_positive    2570  0.015700  0.026127  0.140552
middle_positive   3855  0.006565  0.008579  0.015689
small_positive    6425  0.000001  0.004646  0.006564

 Fractions of overprediction groups made up by different flow regimes
flow_group       high_flow  low_flow  middle_flow
error_group                                      
large_positive       0.129     0.023        0.848
middle_positive      0.086     0.414        0.500
small_positive       0.080     0.470        0.451


Interesting, it seems the highest overpredictions (top 20%) are largely (85%) made up of medium streamflow values (middle 50% of streamflow  values -in between 25th and 75th pecentiles)! This may just be the nature of the model, a result of the poison, or a combination of both. 

In [39]:
# Now we move on to Analysis B: distinguishing severe overprediction from near-accurate prediction within low-flow conditions.
# Only focus on the predictions on low ground truth streamflow that are very close to ground truth or very far away (severe overpredction)

# Analysis B: low-flow accurate vs severe overprediction

low = results.loc[
    results["flow_group"] == "low_flow"
].copy()

# Define near-accuracy relative to the low-flow error distribution.
epsilon_low = low["absolute_residual"].quantile(0.10)

# Define severe overprediction using the upper 20% of
# positive residuals within the low-flow group.
q80_low = low.loc[
    low["residual"] > 0, "residual"
].quantile(0.80)

# Start by excluding all samples.
low["comparison_group"] = "exclude"

# Near-accurate low-flow samples.
low.loc[
    low["absolute_residual"] <= epsilon_low,
    "comparison_group"
] = "low_flow_accurate"

# Severe low-flow overpredictions.
low.loc[
    low["residual"] >= q80_low,
    "comparison_group"
] = "low_flow_overprediction"

print(f"Low-flow accuracy tolerance: {epsilon_low:.6f}")
print(f"Low-flow 80th-percentile residual: {q80_low:.6f}")

print("\nGroup sizes:")
print(low["comparison_group"].value_counts())

print("\nResidual summary:")
print(
    low.groupby("comparison_group")["residual"]
    .agg(["count", "min", "median", "max"])
)

Low-flow accuracy tolerance: 0.004529
Low-flow 80th-percentile residual: 0.007211

Group sizes:
comparison_group
exclude                    3269
low_flow_overprediction     935
low_flow_accurate           468
Name: count, dtype: int64

Residual summary:
                         count       min    median       max
comparison_group                                            
exclude                   3269  0.004529  0.005966  0.007211
low_flow_accurate          468  0.003649  0.004184  0.004529
low_flow_overprediction    935  0.007211  0.007595  0.042798


In [40]:
# Lastly, we create our group sets for each Analysis:

# Analysis C: Low flow vs high flow
groups_C = {
    "low_flow": results.index[
        results["flow_group"] == "low_flow"
    ].to_numpy(),
    "high_flow": results.index[
        results["flow_group"] == "high_flow"
    ].to_numpy(),
}

# Analysis A: Small vs large positive residuals
groups_A = {
    "small_positive": positive.index[
        positive["error_group"] == "small_positive"
    ].to_numpy(),
    "large_positive": positive.index[
        positive["error_group"] == "large_positive"
    ].to_numpy(),
}

# Analysis B: Low-error vs severe low-flow overprediction
groups_B = {
    "low_flow_low_error": low.index[
        low["comparison_group"] == "low_flow_accurate"
    ].to_numpy(),
    "low_flow_severe_overprediction": low.index[
        low["comparison_group"] == "low_flow_overprediction"
    ].to_numpy(),
}

# Verify group size:
for analysis_name, groups in [
    ("C", groups_C),
    ("A", groups_A),
    ("B", groups_B),
]:
    print(f"\nAnalysis {analysis_name}")
    for group_name, idx in groups.items():
        print(f"  {group_name}: {len(idx)} samples")


Analysis C
  low_flow: 4672 samples
  high_flow: 4672 samples

Analysis A
  small_positive: 6425 samples
  large_positive: 2570 samples

Analysis B
  low_flow_low_error: 468 samples
  low_flow_severe_overprediction: 935 samples


#### Now that we have our analysis groups, we can access the activation vectors (post relu) from the model's hidden layers.

In [41]:
# Extract hidden activations using the existing NumPy forward pass.
y_numpy, activations = S.forward(
    w,
    X,
    return_hidden=True
)

# Compare predictions from both implementations.
max_diff = np.max(np.abs(y - y_numpy))

print("Maximum prediction difference:", max_diff)

# Check the activation shapes.
for name, h in activations.items():
    print(name, h.shape)

Maximum prediction difference: 0.0
fc1 (18685, 56)
fc2 (18685, 56)
fc3 (18685, 56)
fc4 (18685, 56)
fc5 (18685, 56)
fc6 (18685, 56)


In [42]:
# Define the sample indices for each analysis.
# These assume you created groups_C, groups_A, and groups_B earlier.

analysis_groups = {
    "C_flow": groups_C,
    "A_error_severity": groups_A,
    "B_low_flow_error": groups_B,
}

# Hidden activation matrices.
layer_names = [f"fc{i}" for i in range(1, 7)]

# Store detailed per-neuron statistics and layer summaries.
neuron_results = []
layer_results = []

for analysis_name, groups in analysis_groups.items():

    group_names = list(groups.keys())
    idx1 = groups[group_names[0]]
    idx2 = groups[group_names[1]]

    for layer_name in layer_names:

        h = activations[layer_name]

        # Extract the activations for each comparison group.
        a = h[idx1]
        b = h[idx2]

        # Per-neuron means and standard deviations.
        mean_a = a.mean(axis=0)
        mean_b = b.mean(axis=0)

        std_a = a.std(axis=0, ddof=1)
        std_b = b.std(axis=0, ddof=1)

        # Difference: first group minus second group.
        mean_diff = mean_a - mean_b

        # Pooled within-group standard deviation.
        pooled_var = (
            (len(a) - 1) * std_a**2
            + (len(b) - 1) * std_b**2
        ) / (len(a) + len(b) - 2)

        pooled_std = np.sqrt(pooled_var)

        # Standardized effect size, handling zero-variance neurons.
        d = np.divide(
            mean_diff,
            pooled_std,
            out=np.full_like(mean_diff, np.nan),
            where=pooled_std > 1e-12
        )

        # Save statistics for every neuron.
        for j in range(h.shape[1]):
            neuron_results.append({
                "analysis": analysis_name,
                "layer": layer_name,
                "neuron": j,
                "mean_group1": mean_a[j],
                "mean_group2": mean_b[j],
                "mean_difference": mean_diff[j],
                "cohens_d": d[j],
            })

        # Summarize the layer's 56 neurons.
        valid_d = d[np.isfinite(d)]

        layer_results.append({
            "analysis": analysis_name,
            "layer": layer_name,
            "n_group1": len(a),
            "n_group2": len(b),
            "mean_difference_norm": np.linalg.norm(mean_diff),
            "max_abs_cohens_d": (
                np.max(np.abs(valid_d)) if len(valid_d) else np.nan
            ),
            "neurons_abs_d_ge_0.5": np.sum(np.abs(valid_d) >= 0.5),
            "neurons_abs_d_ge_1.0": np.sum(np.abs(valid_d) >= 1.0),
        })

neuron_results = pd.DataFrame(neuron_results)
layer_results = pd.DataFrame(layer_results)

print("Layer summaries:")
print(
    layer_results.round(4).to_string(index=False)
)

Layer summaries:
        analysis layer  n_group1  n_group2  mean_difference_norm  max_abs_cohens_d  neurons_abs_d_ge_0.5  neurons_abs_d_ge_1.0
          C_flow   fc1      4672      4672                0.8865            1.9151                    46                    24
          C_flow   fc2      4672      4672                2.2304            5.2631                    47                    46
          C_flow   fc3      4672      4672                0.4251            1.8920                    46                    46
          C_flow   fc4      4672      4672                0.3032            5.1286                    40                    37
          C_flow   fc5      4672      4672                0.1861            4.6942                    32                    31
          C_flow   fc6      4672      4672                0.0998            5.3651                    29                    29
A_error_severity   fc1      6425      2570                0.8132            1.6471            

In [43]:
# Inspect the 10 neurons with the largest absolute standardized effect sizes in each analysis and layer
top_neurons = (
    neuron_results
    .assign(abs_d=lambda df: df["cohens_d"].abs())
    .sort_values(
        ["analysis", "layer", "abs_d"],
        ascending=[True, True, False]
    )
    .groupby(["analysis", "layer"], sort=False)
    .head(10)
)

for analysis_name in analysis_groups:
    print(f"\n{'=' * 65}")
    print(f"Analysis: {analysis_name}")

    for layer_name in layer_names:
        subset = top_neurons.loc[
            (top_neurons["analysis"] == analysis_name)
            & (top_neurons["layer"] == layer_name)
        ]

        print(f"\n{layer_name}:")
        print(
            subset[
                ["neuron", "mean_difference", "cohens_d"]
            ].round(4).to_string(index=False)
        )


Analysis: C_flow

fc1:
 neuron  mean_difference  cohens_d
     18           0.1132    1.9151
     33           0.1128    1.8868
     45           0.1451    1.6230
     14           0.1419    1.5224
      3           0.1291    1.4906
     52          -0.1438   -1.4395
     22          -0.1873   -1.3232
     27          -0.1567   -1.3117
     21          -0.1601   -1.3059
     38           0.0053    1.3001

fc2:
 neuron  mean_difference  cohens_d
     28           0.0000    5.2631
     30           0.0000    5.2163
     35           0.0000    2.8646
     41           0.0000    2.7955
      1           0.0000    2.6848
     55           0.0000    2.5205
     17           0.0000    2.2825
      0          -0.0000   -2.1915
     38          -0.0489   -1.8941
     26          -0.0828   -1.8612

fc3:
 neuron  mean_difference  cohens_d
      6           0.0000    1.8920
     25           0.0000    1.8707
     19           0.0006    1.8616
      9           0.0000    1.8344
     34           0

In [44]:
# Inspect the neurons with the largest effect sizes in each analysis.
# Check their actual activation scales and how often they are zero.

for analysis_name, groups in analysis_groups.items():
    print(f"\n{'=' * 60}\n{analysis_name}")

    group_names = list(groups.keys())
    idx1, idx2 = groups[group_names[0]], groups[group_names[1]]

    for layer_name in layer_names:
        h = activations[layer_name]
        a, b = h[idx1], h[idx2]

        # Recompute standardized effect sizes.
        mean_diff = a.mean(axis=0) - b.mean(axis=0)
        pooled_var = (
            (len(a) - 1) * a.var(axis=0, ddof=1)
            + (len(b) - 1) * b.var(axis=0, ddof=1)
        ) / (len(a) + len(b) - 2)
        pooled_std = np.sqrt(pooled_var)

        d = np.divide(
            mean_diff,
            pooled_std,
            out=np.full(56, np.nan),
            where=pooled_std > 1e-12
        )

        # Inspect the three largest absolute effect sizes.
        top = np.argsort(np.nan_to_num(np.abs(d), nan=-1))[-3:][::-1]

        for j in top:
            print(
                f"{layer_name}, neuron {j}: "
                f"mean1={a[:, j].mean():.8g}, "
                f"mean2={b[:, j].mean():.8g}, "
                f"pooled_std={pooled_std[j]:.8g}, "
                f"d={d[j]:.3f}, "
                f"zero_frac1={(a[:, j] == 0).mean():.3f}, "
                f"zero_frac2={(b[:, j] == 0).mean():.3f}"
            )


C_flow
fc1, neuron 18: mean1=0.13795468, mean2=0.024705354, pooled_std=0.059133902, d=1.915, zero_frac1=0.003, zero_frac2=0.666
fc1, neuron 33: mean1=0.13982375, mean2=0.026994858, pooled_std=0.059799917, d=1.887, zero_frac1=0.004, zero_frac2=0.656
fc1, neuron 45: mean1=0.1863586, mean2=0.041257877, pooled_std=0.089404084, d=1.623, zero_frac1=0.004, zero_frac2=0.643
fc2, neuron 28: mean1=1.5138356e-09, mean2=9.165612e-11, pooled_std=2.7021541e-10, d=5.263, zero_frac1=0.002, zero_frac2=0.861
fc2, neuron 30: mean1=1.3989853e-07, mean2=1.1328548e-08, pooled_std=2.4647566e-08, d=5.216, zero_frac1=0.002, zero_frac2=0.793
fc2, neuron 35: mean1=5.3232125e-06, mean2=3.4296349e-06, pooled_std=6.610228e-07, d=2.865, zero_frac1=0.000, zero_frac2=0.003
fc3, neuron 6: mean1=4.7587696e-06, mean2=3.8539997e-06, pooled_std=4.7828564e-07, d=1.892, zero_frac1=0.000, zero_frac2=0.004
fc3, neuron 25: mean1=6.3783262e-10, mean2=5.2599169e-10, pooled_std=5.9775421e-11, d=1.871, zero_frac1=0.000, zero_frac2

In [45]:

# Analyze Group B: Checking model output after each layer given each Group B condition (low flow accutate and low flow severe overprediction)
idx_accurate = groups_B["low_flow_low_error"]
idx_severe = groups_B["low_flow_severe_overprediction"]

print("Accurate low-flow samples:", len(idx_accurate))
print("Severe low-flow samples:", len(idx_severe))

layer_names = [f"fc{i}" for i in range(1, 7)]


def run_from_hidden(weights, layer_number, h):
    """
    Continue the forward pass from the activation AFTER
    hidden layer `layer_number` (1 through 6).

    h has shape (N, 56).
    """
    for layer in range(layer_number + 1, 7):
        name = f"fc{layer}"
        h = np.maximum(
            0.0,
            h @ weights[name + ".weight"].T
            + weights[name + ".bias"]
        )

    out = (
        h @ weights["out.weight"].T
        + weights["out.bias"]
    )
    return out[:, 0]


def intervention_for_layer(layer_name):
    """
    Remove the group-mean difference direction from the
    activation after the specified layer.
    """
    layer_number = int(layer_name[-1])
    h = activations[layer_name]

    # Use the low-flow comparison to estimate the direction.
    mean_accurate = h[idx_accurate].mean(axis=0)
    mean_severe = h[idx_severe].mean(axis=0)

    delta = mean_severe - mean_accurate
    norm = np.linalg.norm(delta)

    if norm < 1e-12:
        return None

    v = delta / norm

    # Copy activations so the original network is unchanged.
    h_edit = h.copy()

    # Orthogonally remove the direction from the severe group.
    coeff = h_edit[idx_severe] @ v
    h_edit[idx_severe] -= coeff[:, None] * v[None, :]

    # ReLU outputs should be nonnegative. Clipping here makes
    # the intervention feasible, but changes its interpretation.
    h_edit[idx_severe] = np.maximum(
        h_edit[idx_severe], 0.0
    )

    # Original predictions and edited predictions.
    original = y_numpy[idx_severe]
    edited = run_from_hidden(
        w, layer_number, h_edit[idx_severe]
    )

    return {
        "layer": layer_name,
        "direction_norm": norm,
        "mean_original_prediction": original.mean(),
        "mean_edited_prediction": edited.mean(),
        "mean_prediction_change": (edited - original).mean(),
        "fraction_predictions_decreased": np.mean(
            edited < original
        ),
        "mean_original_residual": (
            original - truth[idx_severe]
        ).mean(),
        "mean_edited_residual": (
            edited - truth[idx_severe]
        ).mean(),
    }


intervention_results = []

for layer_name in layer_names:
    result = intervention_for_layer(layer_name)
    if result is not None:
        intervention_results.append(result)

intervention_df = pd.DataFrame(intervention_results)

print(
    intervention_df.round(6).to_string(index=False)
)

Accurate low-flow samples: 468
Severe low-flow samples: 935
layer  direction_norm  mean_original_prediction  mean_edited_prediction  mean_prediction_change  fraction_predictions_decreased  mean_original_residual  mean_edited_residual
  fc1        0.066227                   0.01057                0.014997                0.004427                        0.141176                0.009156              0.013583
  fc2        0.112362                   0.01057                0.007983               -0.002587                        0.550802                0.009156              0.006569
  fc3        0.024900                   0.01057                0.007812               -0.002758                        1.000000                0.009156              0.006398
  fc4        0.017737                   0.01057                0.007781               -0.002789                        1.000000                0.009156              0.006367
  fc5        0.010914                   0.01057                0.01315

The fc6 edit increases the mean prediction from 0.010570 to 0.020537 during Analysis B (low streamflow accurate/low error vs. low streamflow high error)! We should investigate it!

In [46]:
# Most of the high overprediction values (85%) come form mid-range streamflow values so let's investigate that.

# These arrays must be aligned row-for-row with X and activations.
truth = np.asarray(truth)
prediction = np.asarray(y_numpy)
residual = prediction - truth

# Use the same flow thresholds and residual-group definitions
# as in your earlier analyses.
q25, q50, q75 = np.quantile(truth, [0.25, 0.50, 0.75])

flow_group = np.full(len(truth), "middle_flow", dtype=object)
flow_group[truth <= q25] = "low_flow"
flow_group[truth >= q75] = "high_flow"

positive = residual > 0
positive_residuals = residual[positive]

r80 = np.quantile(positive_residuals, 0.80)

error_group = np.full(len(truth), "other", dtype=object)
error_group[positive & (residual <= np.median(positive_residuals))] = "small_positive"
error_group[positive & (residual >= r80)] = "large_positive"

# Targeted groups
idx_mid_large = np.where(
    (flow_group == "middle_flow") &
    (error_group == "large_positive")
)[0]

idx_mid_small = np.where(
    (flow_group == "middle_flow") &
    (error_group == "small_positive")
)[0]

print("Middle-flow large overpredictions:", len(idx_mid_large))
print("Middle-flow small overpredictions:", len(idx_mid_small))

Middle-flow large overpredictions: 2180
Middle-flow small overpredictions: 2895


In [47]:
middle_flow_analysis_results = []

for layer in [f"fc{i}" for i in range(1, 7)]:
    H = activations[layer]

    A = H[idx_mid_large]
    B = H[idx_mid_small]

    mean_A = A.mean(axis=0)
    mean_B = B.mean(axis=0)

    norm_A = np.linalg.norm(mean_A)
    norm_B = np.linalg.norm(mean_B)

    cosine = (
        np.dot(mean_A, mean_B) /
        (norm_A * norm_B + 1e-12)
    )

    pooled_var = (
        (len(A) - 1) * A.var(axis=0, ddof=1)
        + (len(B) - 1) * B.var(axis=0, ddof=1)
    ) / max(len(A) + len(B) - 2, 1)

    pooled_std = np.sqrt(pooled_var)

    d = np.divide(
        mean_A - mean_B,
        pooled_std,
        out=np.zeros_like(mean_A),
        where=pooled_std > 1e-12
    )

    middle_flow_analysis_results.append({
        "layer": layer,
        "mean_difference_norm": np.linalg.norm(mean_A - mean_B),
        "mean_vector_cosine": cosine,
        "max_abs_cohens_d": np.max(np.abs(d)),
        "neurons_abs_d_ge_1": np.sum(np.abs(d) >= 1),
    })

print(pd.DataFrame(middle_flow_analysis_results).round(4).to_string(index=False))

layer  mean_difference_norm  mean_vector_cosine  max_abs_cohens_d  neurons_abs_d_ge_1
  fc1                0.6580              0.5183            1.9140                  21
  fc2                1.6788              0.9997            2.7294                  46
  fc3                0.3176              0.9990            2.5079                  46
  fc4                0.2264              0.9427            3.1119                  37
  fc5                0.1386              0.7574            3.4506                  31
  fc6                0.0738              0.7613            3.2659                  29


In [54]:
layer_names = [f"fc{i}" for i in range(1, 7)]

neuron_rows = []

for layer in layer_names:
    H = activations[layer]

    A = H[idx_mid_large]
    B = H[idx_mid_small]

    mean_diff = A.mean(axis=0) - B.mean(axis=0)

    pooled_var = (
        (len(A) - 1) * A.var(axis=0, ddof=1)
        + (len(B) - 1) * B.var(axis=0, ddof=1)
    ) / (len(A) + len(B) - 2)

    pooled_std = np.sqrt(pooled_var)

    d = np.divide(
        mean_diff,
        pooled_std,
        out=np.full(H.shape[1], np.nan),
        where=pooled_std > 1e-12
    )

    for j in range(H.shape[1]):
        neuron_rows.append({
            "layer": layer,
            "neuron": j,
            "mean_difference": mean_diff[j],
            "cohens_d": d[j],
            "mean_large_error": A[:, j].mean(),
            "mean_small_error": B[:, j].mean(),
            "zero_frac_large": (A[:, j] == 0).mean(),
            "zero_frac_small": (B[:, j] == 0).mean(),
        })

neuron_df = pd.DataFrame(neuron_rows)

top = (
    neuron_df.assign(abs_d=lambda x: x["cohens_d"].abs())
    .sort_values(["layer", "abs_d"], ascending=[True, False])
    .groupby("layer", sort=False)
    .head(5)
)

print(top.round(6).to_string(index=False))

layer  neuron  mean_difference  cohens_d  mean_large_error  mean_small_error  zero_frac_large  zero_frac_small    abs_d
  fc1      52         0.120515  1.914007          0.132786          0.012271         0.015138         0.548187 1.914007
  fc1      22         0.149507  1.793171          0.164649          0.015142         0.015596         0.641451 1.793171
  fc1      27         0.125556  1.703911          0.136170          0.010614         0.025229         0.661140 1.703911
  fc1       4         0.148449  1.648156          0.167015          0.018567         0.098624         0.516753 1.648156
  fc1      25         0.131382  1.508858          0.150122          0.018740         0.070642         0.574784 1.508858
  fc2      30        -0.000000 -2.729409          0.000000          0.000000         0.855046         0.059413 2.729409
  fc2      55        -0.000011 -2.694420          0.000043          0.000054         0.000000         0.000000 2.694420
  fc2      49         0.087959  2.607481

In [ ]:
# Here we check that our forward pass implementation and prediction arrays are consisent and not the result of overpredictions.

print("Max prediction discrepancy:",
      np.max(np.abs(y_numpy - results["prediction"].to_numpy())))

print("Truth alignment:",
      np.max(np.abs(truth - results["truth"].to_numpy())))

# Recompute predictions directly from the loaded model.
with torch.no_grad():
    y_check = net(
        torch.as_tensor(X, dtype=torch.float32)
    ).squeeze(-1).cpu().numpy()

print(
    "Max discrepancy vs y_numpy:",
    np.max(np.abs(y_check - y_numpy))
)

print(
    "Max discrepancy vs results:",
    np.max(np.abs(y_check - results["prediction"].to_numpy()))
)

Max prediction discrepancy: 0.0
Truth alignment: 0.0
Max discrepancy vs y_numpy: 0.0
Max discrepancy vs results: 0.0


In [52]:
# Establish baseline values again

truth_arr = np.asarray(truth).reshape(-1)
pred_arr = np.asarray(y_numpy).reshape(-1)

residual = pred_arr - truth_arr

print("Mean residual:", residual.mean())
print("Mean absolute error:", np.abs(residual).mean())
print("Overprediction fraction:", np.mean(residual > 0))
print("Low-flow overprediction fraction:",
      np.mean(residual[truth_arr <= np.quantile(truth_arr, 0.25)] > 0))

Mean residual: 0.003211961
Mean absolute error: 0.011549565
Overprediction fraction: 0.6877174203906877
Low-flow overprediction fraction: 1.0
